# ETL – Exportaciones DANE (Departamento Administrativo Nacional de Estadística de Colombia)

Este notebook implementa un pipeline ETL para **descargar**, **extraer**, **convertir**, **consolidar** y **limpiar** los datos de exportaciones de bienes de Colombia, correspondientes al período **enero de 2021 – octubre de 2025**.  
El flujo de procesamiento sigue el esquema:

**ZIP / XLSX mensuales → CSV intermedios → dataset analítico limpio (Parquet)**.

## Contexto

Las estadísticas oficiales de exportaciones de bienes en Colombia son producidas por el **DANE**, a partir de los **registros administrativos de las Declaraciones de Exportación (DEX)** presentadas por los exportadores ante la **DIAN**, a través del sistema MUISCA.  
Cada DEX constituye la **unidad básica de observación**, y una vez la autoridad aduanera realiza el cierre de la declaración, esta información es validada, depurada y consolidada por el DANE para la producción de las estadísticas oficiales de comerci.{index=0}.

Si bien el DANE publica estas estadísticas con periodicidad **mensual**, la información se encuentra distribuida en **archivos independientes por período**, sin contar con:
- una **API pública**, ni
- un **dataset agregado y listo para análisis longitudinal** que facilite la exploración histórica de las operaciones de exportación a nivel micro (operación) o meso (producto, país, departamento, aduana, modalidad).

Esto dificulta el uso directo de la información para análisis exploratorios, visualización avanzada, modelamiento estadístico o entrenamiento de modeítica predictiva.

## Objetivo del proyecto

El objetivo principal de este proyecto es **diseñar y documentar un proceso reproducible de consolidación y transformación de datos de exportaciones**, que permita:

- Integrar los reportes mensuales publicados por el DANE en un **único dataset histórico coherente**.
- Aplicar reglas de **limpieza, estandarización y control de calidad** sobre variables clave (fechas, valores FOB, pesos, códigos geográficos y arancelarios).
- Preservar los **tipos de datos analíticos** y la trazabilidad de las variables conforme a la metodología oficial.
- Generar una **fuente analítica estructurada** que sirva como insumo para:
  - ejercicios de visualización y business intelligence,
  - análisis de comercio exterior y desempeño exportador,
  - y futuros entrenamientos de modelos estadísticos o de machine learning.

Este notebook **no busca reemplazar ni reinterpretar** la metodología oficial del DANE, sino **facilitar el acceso analítico** a la información, respetando las definiciones, clasificaciones y alcances establecidos en la documentación técnica de la operación estadística de exportaciones.

## Documentación de referencia

Para el detalle metodológico, definición de variables, clasificaciones, reglas de validación y alcance estadístico, se recomienda consultar la documentación oficial del DANE:

- *Colombia – Estadísticas de Exportaciones (EXPO)*  
  Dirección de Metodología y Producción Estadística (DIMPE), DANE :contentReference[oaicite:1]{index=1}

Dicho documento describe, entre otros aspectos:
- la estructura y contenido de las declaraciones de exportación (DEX),
- las variables de clasificación y análisis (país de destino, subpartida arancelaria, valor FOB, pesos, modalidad, etc.),
- los procesos de validación, consolidación y control de calidad aplicados por el DANE,
- y el marco conceptual bajo el cual se producen las etadísticas oficiales de comercio exterior.


## Estructura del repo

```
data-analytics-portfolio-nicolas-yepes/
├── README.md                  # landin)
├── projects/
│   └── dian-export-etl/
│       ├── README.md          # README del proyecto
│       ├── notebooks/
│       │   └── ETL_Master_file_github.ipynb
│       ├── src/
│       │   ├── dian_export_sync.py
│       │   ├── dian_extract_zip.py
│       │   ├── convert_xlsx_to_csv.py
│       │   └── consolidar_csv.py
│       ├── data/              # ignorado por git
│       │   └── .gitkeep
│       ├── logs/              # ignorado por git
│       │   └── .gitkeep
│       ├── requirements.txt
│       └── .gitignore

```

## Cómo ejecutar

1. Crear entorno e instalar dependencias:
   - `pip install -r requirements.txt`
2. Asegurar que los módulos propios estén en `src/`.
3. Ejecutar el notebook en orden.


In [3]:
# ------------------------------------------------------------
# Configuración de rutas y entorno del proyecto
# ------------------------------------------------------------
# Este bloque permite que el notebook sea reproducible y
# ejecutable desde distintos puntos del repositorio
# (por ejemplo: desde /notebooks o desde la raíz),
# evitando rutas absolutas y configuraciones manuales.
# ------------------------------------------------------------

from pathlib import Path
from importlib import reload
import sys
import pandas as pd

# Directorio actual desde donde se ejecuta el notebook/script
HERE = Path.cwd().resolve()

# Detección automática de la raíz del repositorio.
# Se asume que la raíz contiene la carpeta "data".
if (HERE / "data").exists():
    REPO_ROOT = HERE
elif (HERE.parent / "data").exists():
    REPO_ROOT = HERE.parent
else:
    # Fallback: usar el directorio actual
    # (ajustar si el proyecto usa otra estructura)
    REPO_ROOT = HERE

# Definición centralizada de directorios del proyecto
DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "raw_zip"          # Archivos ZIP originales
EXTRACTED_DIR = DATA_DIR / "extracted"  # Archivos descomprimidos
CSV_DIR = DATA_DIR / "csv"              # CSV intermedios
CONSOLIDATED_DIR = DATA_DIR / "consolidated"  # CSV consolidados
PROCESSED_DIR = DATA_DIR / "processed"  # Dataset limpio final
LOG_DIR = REPO_ROOT / "logs"             # Logs de ejecución

# Crear carpetas necesarias si no existen
for d in [RAW_DIR, EXTRACTED_DIR, CSV_DIR, CONSOLIDATED_DIR, PROCESSED_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Definición explícita de archivos de salida
OUTPUT_CSV = CONSOLIDATED_DIR / "exportaciones_consolidadas.csv"
OUTPUT_PARQUET = PROCESSED_DIR / "exportaciones_limpias.parquet"

# Añadir carpeta src/ al path para importar módulos propios
SRC_DIR = REPO_ROOT / "src"
if SRC_DIR.exists():
    sys.path.append(str(SRC_DIR))

# Configuración visual de pandas para exploración de datos
pd.set_option("display.max_columns", 200)



## 1) Descarga de ZIPs (RAW)

Este paso descarga los archivos ZIP al directorio `data/raw_zip/`.  
Si ya estan descargados, saltar esta sección.


In [2]:
# Importación dinámica del módulo de descarga DIAN/DANE
# Este bloque importa y recarga dinámicamente el módulo
# encargado de sincronizar los archivos ZIP mensuales
# de exportaciones.

try:
    # Importar el módulo completo
    import dian_export_sync

    # Forzar recarga del módulo para reflejar cambios recientes a "dian_export_sync.py"
    reload(dian_export_sync)

    # Importar la función principal de sincronización que descarga de manera automatica los archivos zip directamente de web DIAN.
    from dian_export_sync import sync_dian_exportaciones

except Exception as e:
    # Error controlado y mensaje claro si el módulo no está disponible
    raise ImportError(
        "No se puede importar 'dian_export_sync'. "
        "Revisar que el archivo exista en ./src/dian_export_sync.py "
        "y que la carpeta 'src/' esté correctamente configurada en el path."
    ) from e

# Descarga incremental de archivos ZIP de exportaciones
# La función identifica y descarga únicamente los archivos
# que no existen localmente, comenzando desde el período
# especificado (año y mes).

nuevos = sync_dian_exportaciones(
    dest_dir=RAW_DIR,    # Carpeta donde se almacenan los ZIP crudos
    desde_anio=2021,     # Año inicial del histórico
    desde_mes=1,         # Mes inicial
    verbose=True         # Logs detallados de la descarga
)

# Resumen de archivos descargados en esta ejecución

print("Nuevos ZIP descargados:")
for f in nuevos:
    # Mostrar solo el nombre del archivo (no el path completo)
    print(" -", Path(f).name if not isinstance(f, Path) else f.name)



🔄 Sincronizando exportaciones 2021-01 → 2026-01

📥 Descargando 01_Exportaciones_2021_Enero.zip...
   ✅ Guardado en: C:\Users\nyepes\data\raw_zip\01_Exportaciones_2021_Enero.zip

📥 Descargando 02_Exportaciones_2021_Febrero.zip...
   ✅ Guardado en: C:\Users\nyepes\data\raw_zip\02_Exportaciones_2021_Febrero.zip

📥 Descargando 03_Exportaciones_2021_Marzo.zip...
   ✅ Guardado en: C:\Users\nyepes\data\raw_zip\03_Exportaciones_2021_Marzo.zip

📥 Descargando 04_Exportaciones_2021_Abril.zip...
   ✅ Guardado en: C:\Users\nyepes\data\raw_zip\04_Exportaciones_2021_Abril.zip

📥 Descargando 05_Exportaciones_2021_Mayo.zip...
   ✅ Guardado en: C:\Users\nyepes\data\raw_zip\05_Exportaciones_2021_Mayo.zip

📥 Descargando 06_Exportaciones_2021_Junio.zip...
   ✅ Guardado en: C:\Users\nyepes\data\raw_zip\06_Exportaciones_2021_Junio.zip

📥 Descargando 07_Exportaciones_2021_Julio.zip...
   ✅ Guardado en: C:\Users\nyepes\data\raw_zip\07_Exportaciones_2021_Julio.zip

📥 Descargando 08_Exportaciones_2021_Agosto.zip

## 2) Extracción de ZIPs a carpeta `extracted/`

Extrae los ZIPs descargados para dejar los XLSX/archivos de trabajo listos para conversión.


In [4]:
# Importación dinámica del módulo de extracción de ZIPs
# Este bloque importa y recarga el módulo responsable de
# descomprimir los archivos ZIP descargados previamente.
# El uso de reload() permite reflejar cambios en el código
# del módulo sin necesidad de reiniciar el kernel.

from importlib import reload

try:
    # Importar el módulo completo
    import dian_extract_zip

    # Forzar la recarga del módulo para desarrollo iterativo
    reload(dian_extract_zip)

    # Importar la función principal de extracción
    from dian_extract_zip import extract_all_zips

except Exception as e:
    # Error controlado con mensaje claro si el módulo no está disponible
    raise ImportError(
        "No se puede importar 'dian_extract_zip'. "
        "Revisar de que el archivo exista en ./src/dian_extract_zip.py "
        "y que la carpeta 'src/' esté correctamente configurada en el path."
    ) from e

# Extracción incremental de archivos ZIP
# Se recorren los ZIP descargados y se extraen los archivos
# contenidos (principalmente XLSX), evitando reprocesar
# archivos ya existentes para mantener idempotencia.

nuevas = extract_all_zips(
    zip_dir=str(RAW_DIR),         # Carpeta con ZIP crudos descargados
    extract_dir=str(EXTRACTED_DIR),  # Carpeta destino de archivos extraídos
    overwrite=False,              # No sobrescribir archivos ya extraídos
    verbose=True                  # Logging detallado del proceso
)

# Resumen de la extracción

print(f"ZIPs procesados: {len(nuevas) if nuevas is not None else 'N/D'}")



📦 Extrayendo: 01_Exportaciones_2021_Enero.zip
   → Carpeta destino: C:\Users\nyepes\data\extracted\01_Exportaciones_2021_Enero
   ✅ Extraído correctamente

📦 Extrayendo: 01_Exportaciones_2022_Enero.zip
   → Carpeta destino: C:\Users\nyepes\data\extracted\01_Exportaciones_2022_Enero
   ✅ Extraído correctamente

📦 Extrayendo: 01_Exportaciones_2023_Enero.zip
   → Carpeta destino: C:\Users\nyepes\data\extracted\01_Exportaciones_2023_Enero
   ✅ Extraído correctamente

📦 Extrayendo: 01_Exportaciones_2024_Enero.zip
   → Carpeta destino: C:\Users\nyepes\data\extracted\01_Exportaciones_2024_Enero
   ✅ Extraído correctamente

📦 Extrayendo: 01_Exportaciones_2025_Enero.zip
   → Carpeta destino: C:\Users\nyepes\data\extracted\01_Exportaciones_2025_Enero
   ✅ Extraído correctamente

📦 Extrayendo: 02_Exportaciones_2021_Febrero.zip
   → Carpeta destino: C:\Users\nyepes\data\extracted\02_Exportaciones_2021_Febrero
   ✅ Extraído correctamente

📦 Extrayendo: 02_Exportaciones_2022_Febrero.zip
   → Carpet

## 3) Conversión de XLSX a CSV

Convierte los archivos extraídos a CSV en `data/csv/`.


In [ ]:
# Importación dinámica del módulo de conversión XLSX → CSV
# Este bloque importa y recarga el módulo encargado de convertir
# los archivos XLSX extraídos desde los ZIP en archivos CSV
# estandarizados. Este paso permite abandonar formatos
# ofimáticos y preparar los datos para consolidación y análisis.

from importlib import reload

try:
    # Importar el módulo completo
    import convert_xlsx_to_csv

    # Forzar recarga del módulo para reflejar cambios recientes
    reload(convert_xlsx_to_csv)

    # Importar la función principal de conversión
    from convert_xlsx_to_csv import convertir_xlsx_nuevos_a_csv

except Exception as e:
    # Error controlado con mensaje claro si el módulo no está disponible
    raise ImportError(
        "No se pudo importar 'convert_xlsx_to_csv'. "
        "Revisar que el archivo exista en ./src/convert_xlsx_to_csv.py "
        "y que la carpeta 'src/' esté correctamente configurada en el path."
    ) from e

# Conversión incremental de archivos XLSX a CSV
# Se recorren los archivos XLSX extraídos y se convierten a CSV,
# aplicando una estructura homogénea que facilita la
# consolidación posterior. El proceso registra logs para
# trazabilidad y debugging.

convertir_xlsx_nuevos_a_csv(
    extracted_dir=EXTRACTED_DIR,
    csv_dir=CSV_DIR,
    log_dir=LOG_DIR,
    verbose=True
)

Log de control: C:\Users\nyepes\logs\xlsx_to_csv.log
Archivos ya convertidos: 0
XLSX encontrados en disco: 58
XLSX nuevos a procesar: 58

Convirtiendo 01_Exportaciones_2021_Enero.xlsx...
   CSV guardado en: C:\Users\nyepes\data\csv\01_Exportaciones_2021_Enero.csv

Convirtiendo 01_Exportaciones_2022_Enero.xlsx...
   CSV guardado en: C:\Users\nyepes\data\csv\01_Exportaciones_2022_Enero.csv

Convirtiendo 01_Exportaciones_2023_Enero.xlsx...
   CSV guardado en: C:\Users\nyepes\data\csv\01_Exportaciones_2023_Enero.csv

Convirtiendo 01_Exportaciones_2024_Enero.xlsx...
   CSV guardado en: C:\Users\nyepes\data\csv\01_Exportaciones_2024_Enero.csv

Convirtiendo 01_Exportaciones_2025_Enero.xlsx...
   CSV guardado en: C:\Users\nyepes\data\csv\01_Exportaciones_2025_Enero.csv

Convirtiendo 02_Exportaciones_2021_Febrero.xlsx...
   CSV guardado en: C:\Users\nyepes\data\csv\02_Exportaciones_2021_Febrero.csv

Convirtiendo 02_Exportaciones_2022_Febrero.xlsx...
   CSV guardado en: C:\Users\nyepes\data\csv\

## 4) Consolidación de CSVs

Une todos los CSV generados en un único archivo en `data/consolidated/`.


In [ ]:
# Importación dinámica del módulo de consolidación de CSV
# Este bloque carga el módulo que consolida los CSV mensuales en un único archivo histórico (OUTPUT_CSV)

try:
    # Importar el módulo completo (para poder recargarlo sin reiniciar kernel)
    import consolidar_csv
    reload(consolidar_csv)

    # Importar la función pública principal del módulo
    from consolidar_csv import consolidar_csv_nuevos

except Exception as e:
    raise ImportError(
        "No pude importar 'consolidar_csv'. "
        "Asegúrate de tener el archivo en ./src/consolidar_csv.py"
    ) from e


consolidar_csv_nuevos(
    csv_dir=CSV_DIR,          # Carpeta que contiene los CSV intermedios (provenientes de XLSX)
    output_csv=OUTPUT_CSV,    # Archivo de salida consolidado (unión de períodos)
    log_file=None,
    verbose=True,
)

# Confirmación del artefacto generado
print("Consolidado creado en:", OUTPUT_CSV)

## 5) Carga del consolidado + revisión rápida

En esta sección hacemos un primer *sanity check*: tamaño, columnas y nulos.


In [ ]:
bd = pd.read_csv(OUTPUT_CSV, low_memory=False)
print(f"La dimensión de la base es: {bd.shape}")
bd.head()

In [ ]:
list(bd.columns)

In [ ]:
print(f"Los valores nulos por columna son:\n{bd.isnull().sum()}")

## 6) Limpieza inicial: columnas basura, nulos y duplicados

- Eliminación de columnas `Unnamed:*` (típicas al exportar desde Excel).
- Remoción de filas completamente nulas (si aplica).
- Detección y eliminación de duplicados.


In [ ]:
BD_SIN_NULL = bd.copy()

# Quita columnas tipo Unnamed
BD_SIN_NULL = BD_SIN_NULL.loc[:, ~BD_SIN_NULL.columns.str.contains("^Unnamed")]

print(BD_SIN_NULL.shape)

In [ ]:
duplicados = BD_SIN_NULL[BD_SIN_NULL.duplicated(keep=False)]
duplicados.shape

In [ ]:
BD_SIN_DUP = BD_SIN_NULL.drop_duplicates()
BD_SIN_DUP.shape

## 7) Tipos de datos y preparación (especialmente fechas y códigos)

Aquí revisamos dtypes y estandarizamos:
- Fechas a `datetime64[ns]` cuando aplique.
- Códigos/IDs a `string` (para no perder ceros a la izquierda).


In [ ]:
pd.set_option("display.max_rows", None)

BD_SIN_DUP.dtypes.reset_index(name="dtype").rename(columns={"index": "columna"})

In [ ]:
# Muestra del contenido de columnas de fecha (útil para detectar formatos int/str)
cols_fecha = [
    "FECHA_PROCESO",
    "FECH_DECLA_EXPORTACION_ANT",
    "FECH_DECLA_PRECEDENTE",
    "FECHA_SOLICITUD_AUTO_EMBARQUE",
    "FECHA_DECLARACION_EXPORTACION",
]

cols_fecha_presentes = [c for c in cols_fecha if c in BD_SIN_DUP.columns]
BD_SIN_DUP[cols_fecha_presentes].head()

In [ ]:
#Selección de fecha del la BD y ajuste de formato.
#Teniendo en cuenta que FECHA_DECLARACION_EXPORTACION es la fecha oficial y más granular en cada una de las operaciones de exportación
#se toma esta como la variable fecha guía y se eliminar el resto de columnas
BD_FORM_CORR = (
    BD_SIN_DUP
    .drop(columns=[
        "FECHA_PROCESO",
        "FECH_DECLA_EXPORTACION_ANT",
        "FECH_DECLA_PRECEDENTE",
        "FECHA_SOLICITUD_AUTO_EMBARQUE"
    ])
    .assign(
        FECHA_DECLARACION_EXPORTACION=lambda df: pd.to_datetime(
            df["FECHA_DECLARACION_EXPORTACION"]
            .replace(0, pd.NA)
            .astype(str),
            format="%Y%m%d",
            errors="coerce"
        )
    )
)

In [ ]:
#Ajuste de formato de variables codigo que no tienen un valor númerico real

cols_to_string = [
    "NUMERO_SERIE",
    "OFICINA",
    "COD_ADUANA_DESPACHO",
    "TIPO_IDENT",
    "TIPO_USUARIO",
    "COD_USUARIO",
    "CLASE_EXPORTADOR",
    "COD_DPTO_EXPORTADOR",
    "COD_PAIS_DESTINO_NUM",
    "COD_LUGAR_SALIDA_NUM",
    "COD_REGION_PROCEDENCIA",
    "NUM__DECLA_EXPORTACION_ANT",
    "NUM_DECLARACION_PRECEDENTE",
    "COD_MODO_TRANSPORTE",
    "NIT_EXPORTADOR",
    "NIT_DECLARANTE",
    "BANDERA",
    "COD_REGIMEN_CAN",
    "COD_MODALIDAD_EXPORTACION",
    "FORMA_PAGO",
    "COD_TIPO_EMBARQUE",
    "COD_TIPO_DATOS",
    "TIPO_CERTIFICADO_ORIGEN",
    "SUBPARTIDA",
    "COD_REGION_ORIGEN",
    "COD_UNIDAD_FISICA_NUM",
    "COD_ADUANA_SALIDA",
    "NUMERO_FORMULARIO",
]

BD_FORM_CORR[cols_to_string] = (
    BD_FORM_CORR[cols_to_string]
    .astype("string")
)

In [ ]:
#Ajuste del formato de valores númericos
pd.options.display.float_format = '{:,.2f}'.format

BD_FORM_CORR[
    [
        "CANTIDAD_UNIDADES_FISICAS",
        "PESO_BRUTO_KGS",
        "PESO_NETO_KGS",
        "VALOR_FOB_USD",
        "VALOR_FOB_PESOS",
        "VLR_SERIE_AGREGADO_NAL_USD",
        "VALOR_SERIE_FLETES_USD",
        "VALOR_SERIE_SEGUROS_USD",
        "VLR_SERIE_OTROS_GASTOS_USD",
    ]
].head()

## 8) Export final (para análisis / modelos)

Guardamos el dataset limpio en **Parquet** (más eficiente que CSV) y opcionalmente en CSV.


In [ ]:
# Guarda en Parquet (recomendado)
BD_FORM_CORR.to_parquet(OUTPUT_PARQUET, index=False)
print("✅ Dataset limpio guardado en:", OUTPUT_PARQUET)